# 0. Preparação e Upload dos Dados

## 0.1 - Parâmetros do ambiente

In [0]:
# Padronização de nomes: <catalog>.<camada>.<tabela>, tudo em snake_case
catalog = "cine_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

catalog: cine_medallion
bronze_schema: cine_medallion.bronze
silver_schema: cine_medallion.silver
gold_schema: cine_medallion.gold
landing_path: /Volumes/cine_medallion/bronze/Landing


## 0.2 - Criação do Catalog, Schemas e Volume

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.landing")

print("Catalog, schemas e volume prontos.")

Catalog, schemas e volume prontos.


## 0.3 - Validação da Landing Zone

In [0]:
expected_files = [
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_info_TMDB_IMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "movies_reviews.csv",
    "cotacoes_dolar.csv",
]

try:
    existing = {f.name for f in dbutils.fs.ls(landing_path)}
except Exception as e:
    existing = set()
    print(f"[ALERTA] Não foi possível listar '{landing_path}'. Faça o upload dos arquivos antes de continuar.\n{e}")

missing = [f for f in expected_files if f not in existing]

if missing:
    print("[PENDENTE] Arquivos ainda não encontrados na landing zone:")
    for f in missing:
        print(f"  - {f}")
else:
    print("[OK] Todos os arquivos esperados estão na landing zone:")
    for f in dbutils.fs.ls(landing_path):
        print(f"  - {f.name} ({f.size/1024:.1f} KB)")

[OK] Todos os arquivos esperados estão na landing zone:
  - cotacoes_dolar.csv (0.4 KB)
  - credits_and_tags_IMDB_TMDB.csv (21797.5 KB)
  - movies_financials_IMDB_TMDB.csv (1433.3 KB)
  - movies_info_TMDB_IMDB.csv (33129.0 KB)
  - movies_metrics_IMDB_TMDB.csv (3246.0 KB)
  - movies_reviews.csv (1878.4 KB)


# 1 · Ingestão Bronze

Leitura pura da Landing + gravação em Delta na Bronze, sem nenhuma regra de negócio.
O `ingestion_timestamp` é adicionado **na mesma cadeia que termina em `.write()`**, para reforçar
que, por causa da Lazy Evaluation do Spark, esse timestamp só é calculado quando a ação de
escrita realmente dispara — não no momento da leitura.

In [0]:
from pyspark.sql.functions import current_timestamp

# 1. Mapeamento dos caminhos no Volume (criado no notebook 00, schema landing)
tb_movies_info = f"{landing_path}/movies_info_TMDB_IMDB.csv"
tb_movies_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
tb_movies_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
tb_credits_and_tags = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
tb_movies_reviews = f"{landing_path}/movies_reviews.csv"
tb_dolar_october = f"{landing_path}/cotacoes_dolar.csv"

# 2. Leitura pura (sem transformações de negócio)
df_info_raw = spark.read.csv(tb_movies_info, header=True, inferSchema=True)
df_financials_raw = spark.read.csv(tb_movies_financials, header=True, inferSchema=True)
df_metrics_raw = spark.read.csv(tb_movies_metrics, header=True, inferSchema=True)
df_credits_and_tags_raw = spark.read.csv(tb_credits_and_tags, header=True, inferSchema=True)
df_reviews_raw = spark.read.csv(tb_movies_reviews, header=True,)
df_dolar_october_raw = spark.read.csv(tb_dolar_october, header=True, inferSchema=True)

In [0]:
# 3. Gravação com adição do timestamp no momento da escrita
df_info_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.info")

df_financials_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.financials")

df_metrics_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.metrics")

df_credits_and_tags_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.credits_and_tags")

df_reviews_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.reviews")

df_dolar_october_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.dolar_october")

display(spark.table(f"{catalog}.bronze.info").limit(5))

id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_timestamp
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-10-16T11:12:05.661Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-10-16T11:12:05.661Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-10-16T11:12:05.661Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-10-16T11:12:05.661Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-10-16T11:12:05.661Z
